# 07 Groupby Join - Annotated Notebook

This notebook includes step-by-step markdown sections and English code comments.

## Step 1 - Import required libraries

In [1]:
# Import required libraries.
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

22/02/18 21:41:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Step 2 - Load data into a Spark DataFrame

In [2]:
# Load data into a Spark DataFrame.
df_green = spark.read.parquet('data/pq/green/*/*')

## Step 3 - Register a temporary SQL view

In [3]:
# Register a temporary SQL view.
df_green.registerTempTable('green')

## Step 4 - Run a Spark SQL query

In [18]:
# Run a Spark SQL query.
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

## Step 5 - Run the next processing step

In [26]:
# Run the next processing step.
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

## Step 6 - Load data into a Spark DataFrame

In [20]:
# Load data into a Spark DataFrame.
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.registerTempTable('yellow')

## Step 7 - Run a Spark SQL query

In [22]:
# Run a Spark SQL query.
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

## Step 8 - Run the next processing step

In [27]:
# Run the next processing step.
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

## Step 9 - Load data into a Spark DataFrame

In [46]:
# Load data into a Spark DataFrame.
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

## Step 10 - Run the next processing step

In [47]:
# Run the next processing step.
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

## Step 11 - Run the next processing step

In [48]:
# Run the next processing step.
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

## Step 12 - Write results to storage

In [50]:
# Write results to storage.
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

## Step 13 - Load data into a Spark DataFrame

In [51]:
# Load data into a Spark DataFrame.
df_join = spark.read.parquet('data/report/revenue/total')

## Step 14 - Run the next processing step

In [56]:
# Run the next processing step.
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

## Step 15 - Load data into a Spark DataFrame

In [54]:
# Load data into a Spark DataFrame.
df_zones = spark.read.parquet('zones/')

## Step 16 - Run the next processing step

In [57]:
# Run the next processing step.
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

## Step 17 - Write results to storage

In [62]:
# Write results to storage.
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

## Step 18 - Run this processing step

In [ ]:
# Run this processing step.
